# WEEX V6.47 — MARKET + LIMIT FVG
Wspolna historia OI miedzy wersjami. Automatyczne pobieranie ZIP. Brak automatycznych zlecen.

In [ ]:
import requests
from pathlib import Path
from google.colab import drive, files

# Jedna wspolna historia OI dla wszystkich kolejnych wersji
drive.mount("/content/drive")
folder_oi = Path("/content/drive/MyDrive/WEEX")
folder_oi.mkdir(parents=True, exist_ok=True)
state = folder_oi / "oi_history_shared.json"

# Jednorazowo przenies najnowsza z poprzednich historii
if not state.exists():
    poprzednie = [
        folder_oi / "weex_v645_oi_history_state.json",
        folder_oi / "weex_v646_oi_history_state.json",
        folder_oi / "weex_v644_oi_history_state.json",
    ]
    dostepne = [p for p in poprzednie if p.exists()]
    if dostepne:
        import shutil
        source = max(dostepne, key=lambda p: p.stat().st_mtime)
        shutil.copy2(source, state)
        print("Przywrocono historie OI:", source.name)

# Pobranie kolektora z GitHub
url = "https://raw.githubusercontent.com/danieldziezak1986-hash/weex-futures/main/weex_v647_collector.py"
r = requests.get(url, timeout=30)
r.raise_for_status()
with open("weex_v647_collector.py", "wb") as f:
    f.write(r.content)

# TOP 150 Crypto + TOP 50 TradFi | MARKET + LIMIT
%run weex_v647_collector.py --top 150 --tradfi-top 50 --max-deep 30 --limit-ttl-minutes 180 --state-path /content/drive/MyDrive/WEEX/oi_history_shared.json

# Automatyczne pobranie najnowszego ZIP
folder = Path("/content/weex_v647_results")
archiwa = list(folder.glob("weex_v647_*.zip"))
if not archiwa:
    raise FileNotFoundError("Nie znaleziono ZIP WEEX V6.47")
najnowszy = max(archiwa, key=lambda p: p.stat().st_mtime)
print("Pobieranie wynikow:", najnowszy.name)
files.download(str(najnowszy))
